[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/pydantic-certified/notebooks/day-07-nested-discriminated-unions.ipynb#scrollTo=a1b2c3d4)

---
# Day 7 · Nested Models, Discriminated Unions, and Recursive Types
**certified-journeys / pydantic-certified** · Review · Advanced Composition

> **Goal for today:** Compose complex, multi-level schemas using nested models, discriminated unions, and self-referential recursive types — and understand when and why each pattern applies.


In [ ]:
%pip install -q 'pydantic>=2.0' pydantic-settings fastapi httpx


## Step 1 · Nested Models

Pydantic models can be composed — a field on one model can have another model as its type.
When you pass a plain `dict` where a nested model is expected, Pydantic **coerces it automatically**.

This is the foundation of any real-world schema: addresses inside users,
line items inside orders, configs inside configs.

**Key docs:** https://docs.pydantic.dev/latest/concepts/models/#nested-models

```
Order
├── shipping_address: ShippingAddress
└── items: List[LineItem]
         └── product_id, quantity, unit_price
```


In [ ]:
from __future__ import annotations
from typing import List
from pydantic import BaseModel, Field


class ShippingAddress(BaseModel):
    street: str
    city: str
    country: str = "US"
    zip_code: str


class LineItem(BaseModel):
    product_id: str
    quantity: int = Field(ge=1)          # must be at least 1
    unit_price: float = Field(gt=0)      # must be positive

    @property
    def subtotal(self) -> float:
        return self.quantity * self.unit_price


class Order(BaseModel):
    order_id: str
    shipping_address: ShippingAddress    # nested model
    items: List[LineItem]                # list of nested models

    @property
    def total(self) -> float:
        return sum(item.subtotal for item in self.items)


# Pydantic coerces dicts into nested model instances automatically
raw_data = {
    "order_id": "ORD-001",
    "shipping_address": {               # plain dict — coerced to ShippingAddress
        "street": "123 Main St",
        "city": "Springfield",
        "zip_code": "12345",
    },
    "items": [                          # list of dicts — each coerced to LineItem
        {"product_id": "SKU-A", "quantity": 2, "unit_price": 9.99},
        {"product_id": "SKU-B", "quantity": 1, "unit_price": 24.50},
    ],
}

order = Order.model_validate(raw_data)
print(f"Order ID:  {order.order_id}")
print(f"Ship to:   {order.shipping_address.city}, {order.shipping_address.country}")
print(f"Items:     {len(order.items)}")
print(f"Total:     ${order.total:.2f}")

# Confirm nested objects are proper model instances, not dicts
print(f"\nAddress type: {type(order.shipping_address).__name__}")
print(f"Item[0] type: {type(order.items[0]).__name__}")


### What just happened?

- **Auto-coercion**: raw dicts nested inside the input were automatically converted to `ShippingAddress` and `LineItem` instances — no manual instantiation needed.
- **`Field(ge=1)` / `Field(gt=0)`** enforce numeric constraints at validation time, not at runtime.
- **Properties** like `subtotal` and `total` work naturally on validated model instances.
- **Deep nesting works recursively**: Pydantic traverses the full input tree and validates every level.


## Step 2 · Deep Nesting Validation

Pydantic's validation errors for deeply nested data include the full **field path**,
making debugging straightforward even in complex schemas.

The error location is a tuple of keys/indices that pinpoints exactly where validation failed.


In [ ]:
from pydantic import ValidationError

bad_data = {
    "order_id": "ORD-002",
    "shipping_address": {
        "street": "456 Oak Ave",
        "city": "Shelbyville",
        "zip_code": "67890",
    },
    "items": [
        {"product_id": "SKU-C", "quantity": 0, "unit_price": 5.00},   # quantity=0 → fails ge=1
        {"product_id": "SKU-D", "quantity": 3, "unit_price": -1.00},  # price < 0 → fails gt=0
    ],
}

try:
    Order.model_validate(bad_data)
except ValidationError as e:
    print(f"Found {e.error_count()} validation error(s):\n")
    for err in e.errors():
        loc = " → ".join(str(p) for p in err["loc"])  # full path
        print(f"  [{loc}]  {err['msg']}")


### What just happened?

- **Error paths** like `items → 0 → quantity` pinpoint exactly which nested field failed.
- Pydantic collects **all errors** in one pass — it doesn't stop at the first failure.
- **`e.error_count()`** tells you the total number of failures without iterating.
- The `"loc"` tuple uses integer indices for list items and string keys for model fields.


## Step 3 · Discriminated Unions

A **discriminated union** uses one field (the *discriminator*) to decide which model in a
`Union` to validate against. Pydantic reads that field first and routes directly — no
trial-and-error across each variant.

**Why it matters:** plain `Union[A, B, C]` tries each model left-to-right until one
succeeds. A discriminated union is O(1) dispatch — dramatically faster at scale.

**Docs:** https://docs.pydantic.dev/latest/concepts/unions/

```
Event
├── event_type == 'click'      → ClickEvent
├── event_type == 'scroll'     → ScrollEvent
└── event_type == 'page_view'  → PageViewEvent
```


In [ ]:
from typing import Annotated, Literal, Union
from pydantic import BaseModel, Field


class ClickEvent(BaseModel):
    event_type: Literal["click"]         # discriminator value
    x: int
    y: int
    element_id: str


class ScrollEvent(BaseModel):
    event_type: Literal["scroll"]        # discriminator value
    delta_y: float
    scroll_position: float


class PageViewEvent(BaseModel):
    event_type: Literal["page_view"]     # discriminator value
    url: str
    referrer: str | None = None
    duration_ms: int | None = None


# Annotated union with discriminator field declared
Event = Annotated[
    Union[ClickEvent, ScrollEvent, PageViewEvent],
    Field(discriminator="event_type"),
]


class EventBatch(BaseModel):
    session_id: str
    events: List[Event]   # each dict is dispatched by event_type


raw_batch = {
    "session_id": "sess-xyz-789",
    "events": [
        {"event_type": "page_view", "url": "/home", "referrer": "https://google.com"},
        {"event_type": "click", "x": 320, "y": 180, "element_id": "btn-signup"},
        {"event_type": "scroll", "delta_y": 120.0, "scroll_position": 0.35},
        {"event_type": "click", "x": 640, "y": 400, "element_id": "btn-submit"},
    ],
}

batch = EventBatch.model_validate(raw_batch)
print(f"Session: {batch.session_id}")
for evt in batch.events:
    print(f"  {type(evt).__name__:18s} → {evt.model_dump()}")


### What just happened?

- **`Literal["click"]`** on the `event_type` field tells Pydantic exactly which variant this model corresponds to.
- **`Field(discriminator="event_type")`** on the `Annotated` union activates O(1) dispatch: Pydantic reads `event_type` first and jumps directly to the right class.
- Each event in the list becomes its correct model type — `ClickEvent`, `ScrollEvent`, or `PageViewEvent`.
- **Invalid discriminator values** produce a clean error: `'event_type' is not a valid discriminator value`.


## Step 4 · Discriminated Union Error Messages

When an unknown discriminator value arrives, Pydantic produces a precise error
listing all valid literal values. This is a significant advantage over plain `Union`
which generates confusing multi-model error dumps.


In [ ]:
from pydantic import TypeAdapter

adapter = TypeAdapter(Event)

# Test: valid event
ev = adapter.validate_python({"event_type": "scroll", "delta_y": 50.0, "scroll_position": 0.8})
print(f"Valid event → {type(ev).__name__}: delta_y={ev.delta_y}")

# Test: unknown discriminator value
try:
    adapter.validate_python({"event_type": "hover", "x": 100, "y": 200})
except ValidationError as e:
    print(f"\nUnknown discriminator error:")
    print(f"  {e.errors()[0]['msg']}")

# Test: missing discriminator field entirely
try:
    adapter.validate_python({"x": 100, "y": 200})
except ValidationError as e:
    print(f"\nMissing discriminator error:")
    print(f"  {e.errors()[0]['msg']}")


### What just happened?

- **`TypeAdapter(Event)`** lets you validate a standalone `Annotated` union without wrapping it in a `BaseModel`.
- An unknown discriminator value produces a single, clear error — not a wall of all-model validation failures.
- A **missing** discriminator also errors cleanly, as Pydantic cannot route without it.
- This pattern is ideal for event buses, webhook handlers, and any polymorphic JSON API.


## Step 5 · Recursive Models

A model that references itself requires **postponed annotation evaluation**.
In Pydantic v2 you must call **`model_rebuild()`** after the class is defined,
so Pydantic can resolve the forward reference.

**Docs:** https://docs.pydantic.dev/latest/concepts/postponed_annotations/

Use cases: file-system trees, comment threads, org charts, JSON-schema nodes.

```
TreeNode
├── name: str
├── value: int | None
└── children: List[TreeNode]   ← self-reference
```


In [ ]:
from __future__ import annotations
from typing import List, Optional
from pydantic import BaseModel


class TreeNode(BaseModel):
    name: str
    value: Optional[int] = None
    children: Optional[List[TreeNode]] = None  # forward reference — resolved below


# Required when using self-referential or forward-referenced models
TreeNode.model_rebuild()


tree_data = {
    "name": "root",
    "value": 1,
    "children": [
        {
            "name": "branch-A",
            "value": 2,
            "children": [
                {"name": "leaf-1", "value": 10},
                {"name": "leaf-2", "value": 11},
            ],
        },
        {
            "name": "branch-B",
            "value": 3,
            "children": [
                {
                    "name": "sub-branch",
                    "value": 4,
                    "children": [{"name": "deep-leaf", "value": 42}],
                }
            ],
        },
    ],
}

root = TreeNode.model_validate(tree_data)


def print_tree(node: TreeNode, indent: int = 0) -> None:
    prefix = "  " * indent
    print(f"{prefix}{node.name} (value={node.value})")
    for child in (node.children or []):
        print_tree(child, indent + 1)


print_tree(root)
print(f"\nroot.children[1].children[0].children[0].name = "
      f"{root.children[1].children[0].children[0].name}")


### What just happened?

- **`from __future__ import annotations`** defers evaluation of all annotations, allowing `List[TreeNode]` to be written before `TreeNode` is fully defined.
- **`model_rebuild()`** resolves the forward reference at runtime — without it, validation raises a `PydanticUserError`.
- Deeply nested dicts are coerced recursively: each dict at any depth becomes a `TreeNode` instance.
- **`Optional[List[TreeNode]] = None`** makes leaf nodes valid — they simply have no children.


## Step 6 · Combining Nested, Discriminated, and Recursive Patterns

Real schemas often combine all three techniques. Here we build a minimal
rule-engine schema: rules can be simple comparisons or composite AND/OR
trees — a recursive discriminated union nested inside a config object.


In [ ]:
from __future__ import annotations
from typing import Annotated, List, Literal, Optional, Union
from pydantic import BaseModel, Field


class ComparisonRule(BaseModel):
    kind: Literal["comparison"]
    field: str
    operator: Literal["eq", "ne", "gt", "lt", "gte", "lte", "contains"]
    value: float | int | str | bool


class CompositeRule(BaseModel):
    kind: Literal["and", "or"]
    rules: List[Rule]  # recursive — each element is another Rule


Rule = Annotated[
    Union[ComparisonRule, CompositeRule],
    Field(discriminator="kind"),
]


class RuleSet(BaseModel):
    name: str
    root: Rule  # top-level rule (may nest arbitrarily deep)


# Required to resolve forward reference in CompositeRule.rules
CompositeRule.model_rebuild()
RuleSet.model_rebuild()


rule_data = {
    "name": "premium_user_discount",
    "root": {
        "kind": "and",
        "rules": [
            {"kind": "comparison", "field": "plan", "operator": "eq", "value": "premium"},
            {
                "kind": "or",
                "rules": [
                    {"kind": "comparison", "field": "tenure_months", "operator": "gte", "value": 12},
                    {"kind": "comparison", "field": "spend_total", "operator": "gt", "value": 500},
                ],
            },
        ],
    },
}

ruleset = RuleSet.model_validate(rule_data)
print(f"RuleSet: {ruleset.name}")
print(f"Root kind: {ruleset.root.kind}")
print(f"Root rule count: {len(ruleset.root.rules)}")

inner_or = ruleset.root.rules[1]
print(f"\nInner OR rule has {len(inner_or.rules)} branches:")
for r in inner_or.rules:
    print(f"  {r.field} {r.operator} {r.value}")


### What just happened?

- **Recursive discriminated union**: `CompositeRule.rules` is typed as `List[Rule]`, and `Rule` itself includes `CompositeRule` — a self-referential cycle resolved by `model_rebuild()`.
- **`kind` as discriminator** lets Pydantic route instantly: `"and"` and `"or"` → `CompositeRule`, `"comparison"` → `ComparisonRule`.
- Both `CompositeRule` and `RuleSet` need `model_rebuild()` because they transitively reference the forward `Rule` type.
- This pattern is used in real tools like JSON Schema (`anyOf`, `allOf`), query builders, and policy engines.


In [ ]:
# Challenge: Build a FileSystemNode model
#
# A file system has two node types:
#   - FileNode: kind='file', name: str, size_bytes: int
#   - DirectoryNode: kind='dir', name: str, children: List[FSNode]
#
# Requirements:
#   1. Use a discriminated union FSNode = Annotated[Union[FileNode, DirectoryNode], Field(discriminator='kind')]
#   2. Call model_rebuild() on DirectoryNode
#   3. Validate the sample data below and print the full directory tree
#   4. Add a helper function that counts total files in a tree
#
# Sample data:
sample_fs = {
    "kind": "dir", "name": "project",
    "children": [
        {"kind": "file", "name": "README.md", "size_bytes": 1024},
        {
            "kind": "dir", "name": "src",
            "children": [
                {"kind": "file", "name": "main.py", "size_bytes": 4096},
                {"kind": "file", "name": "utils.py", "size_bytes": 2048},
            ],
        },
    ],
}

# Your solution here
# from __future__ import annotations
# class FileNode(BaseModel): ...
# class DirectoryNode(BaseModel): ...
# FSNode = Annotated[...]
# DirectoryNode.model_rebuild()
# root = TypeAdapter(FSNode).validate_python(sample_fs)


---
## Day 7 key concepts recap

| Concept | What to remember |
|---|---|
| Nested models | Pass a `dict` where a model field is expected — Pydantic coerces automatically |
| Nested error paths | `e.errors()[n]['loc']` gives the full path: `items → 0 → quantity` |
| Discriminated union | `Annotated[Union[A, B], Field(discriminator='x')]` — O(1) dispatch by literal value |
| `Literal["value"]` | Pins a field to a single constant — required on each discriminated variant |
| Recursive models | Use `from __future__ import annotations` + call `model_rebuild()` after the class |
| `TypeAdapter` | Validate a standalone type without wrapping it in `BaseModel` |

> **Tip:** Discriminated unions are dramatically faster than plain Union types because Pydantic reads the discriminator field first and jumps directly to the correct validator.

---
## What's next
**Day 8** → FastAPI integration — how Pydantic models become request bodies, response schemas, and OpenAPI docs automatically.

Mark Day 7 complete in your [tracker](../index.html).
